# Telco Customer Churn Predictor (cleaned)

This notebook trains a churn classifier on the **Telco Customer Churn** dataset.

Key cleanups compared to the original draft:
- **No data leakage**: preprocessing and SMOTE happen **inside** a pipeline and cross-validation.
- Consistent imports, naming, and reproducibility (single `SEED`).
- More robust `TotalCharges` handling (converted to numeric; blanks → 0).
- Clear evaluation (CV + holdout test), with metrics beyond accuracy.


## Load dataset

## Basic cleaning

## Quick EDA (optional)

## Train/test split

## Preprocessing + SMOTE + models (pipelines)

## Cross-validated comparison

## Train best model and evaluate on the holdout test set

## Hyperparameter tuning (example: Random Forest)

## Save the trained pipeline (optional)

In [ ]:
# Core imports
import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


In [ ]:
# If you don't have kagglehub yet, install with:
# !pip install -U kagglehub

import kagglehub

# Download latest version of the dataset from KaggleHub
dataset_dir = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Dataset directory:", dataset_dir)

csv_path = os.path.join(dataset_dir, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw = pd.read_csv(csv_path)
df_raw.head()


In [ ]:
df = df_raw.copy()

# Convert TotalCharges to numeric (dataset sometimes has blanks)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# In this dataset, missing TotalCharges typically happens when tenure == 0.
# Filling with 0 is a reasonable default (alternatively: median imputation).
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Drop ID column
df = df.drop(columns=["customerID"])

# Quick sanity checks
print("Shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).head(10))
df.dtypes


In [ ]:
# Target distribution
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Churn")
plt.title("Churn class distribution")
plt.show()

# A quick look at numeric correlations
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(6, 5))
sns.heatmap(df[num_cols].corr(), annot=True, linewidths=0.5)
plt.title("Correlation heatmap (numeric features)")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

TARGET_COL = "Churn"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean().round(3), "Test churn rate:", y_test.mean().round(3))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import StratifiedKFold, cross_validate

# Identify columns
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object", "category", "bool"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

# Preprocessing: scale numeric, one-hot encode categoricals
# - OneHotEncoder(sparse_output=False) makes downstream SMOTE straightforward.
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="drop",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model(name, model):
    pipe = ImbPipeline(steps=[
        ("preprocess", preprocess),
        ("smote", SMOTE(random_state=SEED)),
        ("model", model),
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "roc_auc": "roc_auc",
            "f1": "f1",
            "precision": "precision",
            "recall": "recall",
        },
        n_jobs=-1,
        return_train_score=False,
    )
    out = {k: float(np.mean(v)) for k, v in scores.items() if k.startswith("test_")}
    out = {k.replace("test_", ""): v for k, v in out.items()}
    out["model"] = name
    return out, pipe



In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent", random_state=SEED),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

# XGBoost is optional
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    )
except Exception as e:
    print("XGBoost not available (skipping). Install with: pip install xgboost")
    print("Reason:", repr(e))

rows = []
pipes = {}

for name, model in models.items():
    row, pipe = evaluate_model(name, model)
    rows.append(row)
    pipes[name] = pipe

results = pd.DataFrame(rows).set_index("model").sort_values(by="roc_auc", ascending=False)
results


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)

best_name = results.index[0]
print("Best model by CV ROC-AUC:", best_name)

best_pipe = pipes[best_name]

# Fit on the full training set
best_pipe.fit(X_train, y_train)

# Predict on test
y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1] if hasattr(best_pipe, "predict_proba") else None

print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title(f"Confusion matrix: {best_name}")
plt.show()

if y_proba is not None:
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title(f"ROC curve: {best_name}")
    plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV

# Only run this if you want tuning; it can take a while.
rf_pipe = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=SEED)),
    ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])

param_grid = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

grid = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

tuned = grid.best_estimator_
y_pred_tuned = tuned.predict(X_test)
print("\nTuned RF test report:")
print(classification_report(y_test, y_pred_tuned, digits=3))


In [ ]:
# This saves the full preprocessing + SMOTE + model pipeline.
# Reload later and call .predict on raw feature columns.

import joblib

MODEL_PATH = "churn_model_pipeline.joblib"
joblib.dump(best_pipe, MODEL_PATH)
print("Saved to:", MODEL_PATH)
